# Part 1 — Ingest, clean, enrich and export
Run in order from repository root. The original data is never modified.

In [ ]:
import csv, json
accepted, rejected = [], []
def parse_row(row):
    converted = row.copy()
    for col in ['age','fee_inr','prior_no_shows']:
        converted[col] = int(converted[col])
    converted['distance_km'] = float(converted['distance_km'])
    return converted
with open('data/appointments.csv', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    print('Header:', reader.fieldnames)
    rows = list(reader)
for row in rows:
    try: accepted.append(parse_row(row))
    except ValueError: rejected.append(row)
with open('rejected_rows.json','w') as f: json.dump(rejected,f,indent=2)
print('Raw',len(rows),'Accepted',len(accepted),'Rejected',len(rejected))

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('data/appointments.csv')
print('Shape',df.shape)
df.info()
print('Nulls\n',df.isnull().sum())
print('Exact duplicates',df.duplicated().sum())
print('Raw channels',df.booking_channel.nunique())
df = df.drop_duplicates().copy()
print('After duplicates',len(df))
df['booking_channel'] = df.booking_channel.str.strip().str.title()
print('Clean channels',sorted(df.booking_channel.unique()))
df['appt_date'] = pd.to_datetime(df.appt_date,errors='coerce')
df['booked_on'] = pd.to_datetime(df.booked_on)
print('Bad dates',df.appt_date.isna().sum(),'Missing status',df.status.isna().sum())
df = df.dropna(subset=['appt_date','status']).copy()
print('After date/status drop',len(df))

In [ ]:
bad_age = (df.age < 1) | (df.age > 100)
print('Invalid ages',bad_age.sum())
df.loc[bad_age,'age'] = np.nan
age_median = df.age.median(); distance_median = df.distance_km.median()
print('Medians',age_median,distance_median)
df['age'] = df.age.fillna(age_median).round().astype(int)
df['distance_km'] = df.distance_km.fillna(distance_median)
for col in ['distance_km','fee_inr']:
    q1,q3 = df[col].quantile([.25,.75]); iqr=q3-q1
    low,high=q1-1.5*iqr,q3+1.5*iqr
    print(col,'Q1',q1,'Q3',q3,'fences',low,high,'capped',((df[col]<low)|(df[col]>high)).sum())
    df[col] = np.clip(df[col],low,high)
df['distance_km'] = df.distance_km.round(1)
df['fee_inr'] = df.fee_inr.round().astype(int)

In [ ]:
df['lead_days'] = (df.appt_date-df.booked_on).dt.days
df['no_show'] = (df.status=='No-Show').astype(int)
X_numpy = df[['age','distance_km','fee_inr','lead_days']].to_numpy()
print('Column means',X_numpy.mean(axis=0))
mask = X_numpy[:,3]>14
print('Lead >14',mask.sum(), 'share',mask.mean())
x=X_numpy[:,1];z=(x-np.mean(x))/np.std(x)
print('Distance abs(z)>2',np.sum(np.abs(z)>2))
print('No shows',df.no_show.sum(),'rate',df.no_show.mean())

In [ ]:
import requests
clinics = pd.read_csv('data/clinics.csv')
def fetch_temp(clinic):
    url='https://api.open-meteo.com/v1/forecast'
    params={'latitude':float(clinic.latitude),'longitude':float(clinic.longitude),
            'hourly':'temperature_2m','forecast_days':1}
    for attempt in range(2):
        try:
            response=requests.get(url,params=params,timeout=10)
            if response.status_code != 200: raise ValueError(f'HTTP {response.status_code}')
            temperatures=response.json()['hourly']['temperature_2m']
            if not temperatures: raise ValueError('empty hourly temperature')
            return temperatures[0]
        except (requests.exceptions.RequestException,ValueError,KeyError,TypeError) as exc:
            print(clinic.clinic_name,'attempt',attempt+1,'failed:',exc)
    return None
clinics['temp_c'] = [fetch_temp(row) for row in clinics.itertuples(index=False)]
clinics.to_csv('clinics_enriched.csv',index=False)
print(clinics[['clinic_name','temp_c']])

In [ ]:
columns=['appointment_id','patient_id','clinic_id','booked_on','appt_date','age','gender',
         'distance_km','fee_inr','booking_channel','visit_type','reminder_sent',
         'prior_no_shows','status','lead_days','no_show']
df=df.sort_values('appointment_id').reset_index(drop=True)
for col in ['booked_on','appt_date']:df[col]=df[col].dt.strftime('%Y-%m-%d')
df=df[columns]
print('Unique IDs',df.appointment_id.nunique(),'rows',len(df))
print('Missing by column',df.isna().sum().to_dict())
assert df.appointment_id.is_unique and not df.isna().any().any()
df.to_csv('appointments_clean.csv',index=False)
def sql_value(v):
    if pd.isna(v):return 'NULL'
    if isinstance(v,(str,np.str_)):return "'"+str(v).replace("'","''")+"'"
    return str(v)
with open('load_data.sql','w') as f:
    for row in clinics.itertuples(index=False,name=None):
        f.write('INSERT INTO clinics VALUES ('+','.join(sql_value(v) for v in row)+');\n')
    for row in df.itertuples(index=False,name=None):
        f.write('INSERT INTO appointments VALUES ('+','.join(sql_value(v) for v in row)+');\n')
print('Wrote 998 INSERT statements')